# Generar Tensor para ConvLSTM (Sit. 3)

1. Descarga tiles S2 desde HF
2. Lee panel S5P desde **GCS** (autenticación Colab) o desde HF si está disponible
3. Carga RemoteCLIP + checkpoint fine-tuned (ruta directa en Drive)
4. Extrae embeddings 512-d
5. Genera secuencias + targets T+1/T+3/T+7
6. Construye tensor (N, 8, 522, 5, 5) y sube a HF

In [ ]:
# @title 0. Instalar dependencias
%pip install -q huggingface_hub open_clip_torch zarr numcodecs pandas pyarrow numpy scipy
%pip install -q torch torchvision tqdm matplotlib xarray gcsfs

In [ ]:
from __future__ import annotations
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip
from huggingface_hub import hf_hub_download, snapshot_download, HfApi
from tqdm.notebook import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Descargar tiles S2 desde HF

In [ ]:
# @title Descargar dataset_sit2
HF_TOKEN = os.environ.get("HF_TOKEN")
DATA_DIR = Path("/content/dataset_sit2")

if not (DATA_DIR / "metadatos.parquet").is_file():
    print("Descargando dataset_sit2...")
    snapshot_download(
        repo_id="Slucu-0310/geovision-cali-sit2",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        local_dir_use_symlinks=False,
        token=HF_TOKEN,
    )
print("OK")

In [ ]:
# @title Cargar tiles Zarr
import zarr

df = pd.read_parquet(DATA_DIR / "metadatos.parquet")
df["fecha_dt"] = pd.to_datetime(df["fecha"])
print(f"Tiles: {len(df)}, fechas: {df['fecha_dt'].min()} a {df['fecha_dt'].max()}")

store = zarr.storage.LocalStore(str(DATA_DIR / "tiles.zarr"))
tiles_z = zarr.open(store, mode="r")
if isinstance(tiles_z, zarr.Group):
    tiles_z = tiles_z["tiles"]
print(f"Tiles Zarr: {tiles_z.shape}")

## 2. Panel S5P (desde GCS vía Colab)

In [ ]:
# @title Autenticar GCP en Colab y cargar panel S5P
import xarray as xr
import gcsfs

GCS_BUCKET = "gs://geovision-cali"

# Autenticar GCP (funciona en Colab automáticamente)
from google.colab import auth
auth.authenticate_user()
print("Autenticado GCP")

# Conectar a GCS
fs = gcsfs.GCSFileSystem(project="proyecto3ia-494900")

# Cargar paneles S5P
s5p_panels = {}
for pol in ["NO2", "SO2", "O3"]:
    path = f"{GCS_BUCKET}/Sentinel5P/{pol}/panel.zarr"
    ds = xr.open_zarr(fs.get_mapper(path), consolidated=True)
    s5p_panels[pol] = ds
    print(f"S5P {pol}: {len(ds.time)} fechas, y={ds.dims['y']}, x={ds.dims['x']}")

## 3. Cargar RemoteCLIP + checkpoint (ruta directa en Drive)

In [ ]:
# @title Montar Drive (checkpoint del modelo)
from google.colab import drive
drive.mount("/content/drive")

# Ruta DIRECTA al checkpoint (sin glob recursivo que se cuelga)
CKPT = Path("/content/drive/MyDrive/geovision-cali/sit2_geovision_clip/best.pt")
if not CKPT.is_file():
    # Fallback: buscar en la otra carpeta
    CKPT = Path("/content/drive/MyDrive/geovision-cali/sit2_sae_posttrain/clip_modelo/best.pt")
if not CKPT.is_file():
    raise FileNotFoundError(
        "No se encuentra best.pt. Debe estar en:\n"
        "  /content/drive/MyDrive/geovision-cali/sit2_geovision_clip/best.pt\n"
        "Ejecuta primero los notebooks de Sit2 para generar los pesos."
    )
print(f"Checkpoint: {CKPT.name} ({CKPT.stat().st_size/1e6:.0f} MB)")

In [ ]:
# @title Cargar RemoteCLIP + pesos fine-tuned
print("Cargando RemoteCLIP (esto descarga ~600MB la primera vez)...")
cm, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained=None)
cp_remote = hf_hub_download("chendelong/RemoteCLIP", "RemoteCLIP-ViT-B-32.pt",
                            cache_dir="/content/checkpoints")
cm.load_state_dict(torch.load(cp_remote, map_location="cpu"), strict=False)

# Adaptar conv1 de 3 a 12 canales
oc = cm.visual.conv1
nc = nn.Conv2d(12, oc.out_channels, oc.kernel_size,
               stride=oc.stride, padding=oc.padding, bias=False)
with torch.no_grad():
    ir, ig, ib = 3, 2, 1
    w = nc.weight.data
    w[:,ir] = oc.weight[:,0]
    w[:,ig] = oc.weight[:,1]
    w[:,ib] = oc.weight[:,2]
    wm = oc.weight.mean(dim=1, keepdim=False)
    for b in range(12):
        if b not in (ir, ig, ib):
            w[:,b] = wm * (3.0/12)
    nc.weight.copy_(w)
cm.visual.conv1 = nc

# Cargar pesos fine-tuned
state = torch.load(CKPT, map_location="cpu")
if "cm" in state:
    cm.load_state_dict(state["cm"], strict=False)
elif "model" in state:
    cm.load_state_dict(state["model"], strict=False)
else:
    cm.load_state_dict(state, strict=False)

cm = cm.to(DEVICE).eval()
print(f"RemoteCLIP listo en {DEVICE}")

In [ ]:
# @title Extraer embeddings 512-d
print("Extrayendo embeddings...")
rng = np.random.default_rng(42)
ix = rng.choice(tiles_z.shape[0], min(512, tiles_z.shape[0]), replace=False)
sm = np.stack([np.asarray(tiles_z[i], dtype=np.float32) for i in ix])
bm = sm.mean(axis=(0,2,3)).astype(np.float32)
bs = np.maximum(sm.std(axis=(0,2,3)), 1e-3).astype(np.float32)

BATCH_SIZE = 64
all_feats = []
with torch.no_grad():
    for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Embeddings 512-d"):
        end = min(i + BATCH_SIZE, len(df))
        batch = np.stack([
            (np.asarray(tiles_z[j], dtype=np.float32) - bm) / bs
            for j in range(i, end)
        ])[:, :12]  # 12 bandas (sin SCL)
        tiles = torch.from_numpy(batch).float().to(DEVICE)
        tiles_224 = F.interpolate(tiles, size=(224,224), mode="bilinear", align_corners=False)
        all_feats.append(F.normalize(cm.visual(tiles_224), dim=-1).cpu().numpy())

embeddings = np.concatenate(all_feats, axis=0)
np.save(DATA_DIR / "embeddings_sit2.npy", embeddings)
print(f"Embeddings 512-d: {embeddings.shape}")

## 4. Secuencias + targets T+1/T+3/T+7

In [ ]:
# @title Generar secuencias y extraer targets S5P
RES = 0.05
SEQ_LEN = 8
HORIZONS = [1, 3, 7]

df["celda_lat"] = (df["centroide_lat"] / RES).round() * RES
df["celda_lon"] = (df["centroide_lon"] / RES).round() * RES
df["embedding"] = [embeddings[i] for i in range(len(embeddings))]

def get_s5p(panel, fecha, lat, lon, max_days=3):
    times = pd.to_datetime(panel["time"].values)
    target = np.datetime64(fecha)
    diffs = np.abs(times - target)
    idx = np.argmin(diffs)
    if diffs[idx] > np.timedelta64(max_days, "D"):
        return np.nan
    yi = np.argmin(np.abs(panel["y"].values - lat))
    xi = np.argmin(np.abs(panel["x"].values - lon))
    val = float(panel["data"].isel(time=idx, band=0, y=yi, x=xi).values)
    return val if np.isfinite(val) and val > 0 else np.nan

sequences = []
for (clat, clon), grp in df.groupby(["celda_lat", "celda_lon"]):
    ord_grp = grp.sort_values("fecha_dt")
    fechas_u = ord_grp["fecha_dt"].unique()
    if len(fechas_u) < SEQ_LEN:
        continue
    for i in range(len(fechas_u) - SEQ_LEN + 1):
        ventana = fechas_u[i:i+SEQ_LEN]
        rows = [ord_grp[ord_grp["fecha_dt"] == f].iloc[0] for f in ventana]
        last_date = ventana[-1]
        targets = []
        for h in HORIZONS:
            tgt = last_date + pd.Timedelta(days=h)
            targets.append([get_s5p(s5p_panels[p], tgt, clat, clon) for p in ["NO2","SO2","O3"]])
        sequences.append({
            "celda_lat": clat, "celda_lon": clon,
            "rows": rows,
            "targets": np.array(targets, dtype=np.float32),
        })
print(f"Secuencias: {len(sequences)}")

## 5. Construir tensor y subir a HF

In [ ]:
# @title Construir X=(N,8,522,5,5) + y=(N,3,3)
C, GS, STRIDE = 522, 5, 0.005

def build_X(rows):
    lat_c, lon_c = rows[0]["centroide_lat"], rows[0]["centroide_lon"]
    half = (GS - 1) / 2
    lats = np.linspace(lat_c - half*STRIDE, lat_c + half*STRIDE, GS)
    lons = np.linspace(lon_c - half*STRIDE, lon_c + half*STRIDE, GS)
    X = np.zeros((len(rows), C, GS, GS), dtype=np.float32)
    for t, row in enumerate(rows):
        emb, fecha = row["embedding"], row["fecha_dt"]
        doy = fecha.dayofyear
        for gi in range(GS):
            for gj in range(GS):
                X[t, :512, gi, gj] = emb
                X[t, 512, gi, gj] = np.sin(2*np.pi*doy/365.25)
                X[t, 513, gi, gj] = np.cos(2*np.pi*doy/365.25)
                X[t, 514, gi, gj] = np.sin(2*np.pi*fecha.month/12)
                X[t, 515, gi, gj] = np.cos(2*np.pi*fecha.month/12)
                X[t, 516, gi, gj] = (lats[gi] - 3.45) / 0.1
                X[t, 517, gi, gj] = (lons[gj] + 76.5) / 0.1
                X[t, 518, gi, gj] = 0.0 if t == 0 else min((rows[t]["fecha_dt"] - rows[t-1]["fecha_dt"]).days / 30.0, 5.0)
                X[t, 519, gi, gj] = row.get("ndvi", 0) if np.isfinite(row.get("ndvi", 0)) else 0
                X[t, 520, gi, gj] = row.get("bsi", 0) if np.isfinite(row.get("bsi", 0)) else 0
                X[t, 521, gi, gj] = row.get("ndbi", 0) if np.isfinite(row.get("ndbi", 0)) else 0
    return X

X_list, y_list = [], []
for seq in tqdm(sequences):
    X_list.append(build_X(seq["rows"]))
    y_list.append(seq["targets"])

X_full = np.stack(X_list, axis=0)
y_full = np.stack(y_list, axis=0)
print(f"X: {X_full.shape}, y: {y_full.shape}")

In [ ]:
# @title Guardar y subir a HF
np.save(DATA_DIR / "X_convlstm.npy", X_full)
np.save(DATA_DIR / "y_convlstm.npy", y_full)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)
SIT3_REPO = "Slucu-0310/geovision-cali-sit3"
api.create_repo(SIT3_REPO, repo_type="dataset", exist_ok=True)
api.upload_file(path_or_fileobj=str(DATA_DIR/"X_convlstm.npy"),
                path_in_repo="X_convlstm.npy", repo_id=SIT3_REPO, repo_type="dataset")
api.upload_file(path_or_fileobj=str(DATA_DIR/"y_convlstm.npy"),
                path_in_repo="y_convlstm.npy", repo_id=SIT3_REPO, repo_type="dataset")
print(f"Subido a https://huggingface.co/datasets/{SIT3_REPO}")